# Assignment 3: Transformer is All You Need

**Task:** Train a small decoder-only Transformer for next-token prediction on Tiny Shakespeare, then visualize its attention and explain its learning behavior.

This notebook implements:

1. Data preparation (positional split, subword tokenizer, token windows)
2. A tiny causal Transformer built from basic PyTorch layers
3. Training, loss curves, perplexity, and attention visualization
4. A controlled comparison of configurations


> **Learning track — Step 0.** Same code as `submission/assign3_step0.ipynb`, plus learning notes, self-checks, and report prompts. Edit the submission notebook first, then mirror here.


## 0. Setup

Import the libraries used below and fix the constants that make the work reproducible. Hardware and package versions are recorded because training time and exact losses depend on them; every configuration is trained in the same runtime so their times are comparable.


In [1]:
import hashlib
import os
import platform
import random
import subprocess
import sys
from datetime import date
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib
import numpy as np
import torch

try:
    import tokenizers
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tokenizers"])
    import tokenizers

assert hasattr(torch.nn, "RMSNorm"), "PyTorch >= 2.4 is required for nn.RMSNorm"

SEED = 42
TRAIN_FRACTION = 0.8
ARTIFACTS_DIR = Path("artifacts")  # tokenizer and selected checkpoint
ASSETS_DIR = Path("assets")  # figures used in the report
ARTIFACTS_DIR.mkdir(exist_ok=True)
ASSETS_DIR.mkdir(exist_ok=True)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def cpu_model():
    if platform.system() == "Linux":
        for line in Path("/proc/cpuinfo").read_text().splitlines():
            if line.startswith("model name"):
                return line.split(":", 1)[1].strip()
    if platform.system() == "Darwin":
        result = subprocess.run(["sysctl", "-n", "machdep.cpu.brand_string"], capture_output=True, text=True)
        return result.stdout.strip()
    return platform.processor() or "unknown"


set_seed(SEED)
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Python      {platform.python_version()}")
print(f"PyTorch     {torch.__version__}")
print(f"tokenizers  {tokenizers.__version__}")
print(f"NumPy       {np.__version__}")
print(f"matplotlib  {matplotlib.__version__}")
print(f"Platform    {platform.platform()}")
print(f"CPU         {cpu_model()} ({os.cpu_count()} cores)")
if DEVICE.type == "cuda":
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU         {gpu.name} ({gpu.total_memory / 1024**3:.1f} GB)")
print(f"Device      {DEVICE}")
print(f"Seed        {SEED}")


Python      3.9.6
PyTorch     2.8.0
tokenizers  0.22.2
NumPy       2.0.2
matplotlib  3.9.4
Platform    macOS-26.6.2-arm64-arm-64bit
CPU         Apple M5 (10 cores)
Device      mps
Seed        42


**Learning notes — Setup**

- `SEED = 42` seeds Python, NumPy, and PyTorch, so weight initialization and batch order repeat across runs.
- `DEVICE` prefers a CUDA GPU (Colab T4), then Apple MPS (local Mac), then CPU. The final reported numbers come from the Colab run, so the report must name that device.
- `nn.RMSNorm` exists only from PyTorch 2.4; the `assert` fails early instead of deep inside the model code.
- `artifacts/` will hold the tokenizer and selected checkpoint (the handout asks for both to be saved); `assets/` holds report figures.

**Self-check**

1. Why must all configurations be trained in the same runtime if their training times are compared?
2. Which results would change if only the seed changed?

**Report prompt:** Methods should list optimizer, batch size, learning rate, steps, parameter count, hardware, package versions, seed, and training time. This cell supplies hardware, versions, and seed.


### 0.1 Load the data

Prefer a local `input.txt` in the notebook working directory (Colab upload). If it is missing, download it from the source, Andrej Karpathy's char-rnn repository. The SHA-256 check guarantees that every run and every experiment uses exactly the same text.


In [2]:
DATA_FILE = Path("input.txt")
DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
EXPECTED_SHA256 = "86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed"

if DATA_FILE.exists():
    # The uploaded file is the repository copy, fetched from DATA_URL on this date.
    download_date = "2026-11-21"
else:
    urlretrieve(DATA_URL, DATA_FILE)
    download_date = date.today().isoformat()

raw_bytes = DATA_FILE.read_bytes()
sha256 = hashlib.sha256(raw_bytes).hexdigest()
assert sha256 == EXPECTED_SHA256, f"Unexpected file contents (SHA-256 {sha256})"
text = raw_bytes.decode("utf-8")
chars = sorted(set(text))

print(f"Source         {DATA_URL}")
print(f"Download date  {download_date}")
print(f"SHA-256        {sha256}")
print(f"Characters     {len(text):,}")
print(f"Lines          {text.count(chr(10)):,}")
print(f"Unique chars   {len(chars)}: {repr(''.join(chars))}")
print(f"Spaces         {text.count(' '):,} ({text.count(' ') / len(text):.1%} of characters)")
print()
print(text[:250])


Source         https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Download date  2026-11-21
SHA-256        86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed
Characters     1,115,394
Lines          40,000
Unique chars   65: "\n !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"
Spaces         169,892 (15.2% of characters)

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.



**Interpretation.**

- The file matches the expected checksum: 1,115,394 characters in 40,000 lines, so every experiment below uses identical text.
- Only 65 distinct characters occur: the 52 upper- and lower-case letters, space, newline, the digit `3`, and ten punctuation marks. Spaces alone are about 15% of the text.
- The corpus is play dialogue: a speaker name on its own line ending in a colon, followed by the speech. Newlines, colons, and capitalized names are therefore frequent structural patterns that the tokenizer and model must represent.


**Learning notes — Data**

- The handout requires the **source** and **download date**; the SHA-256 hash proves the same text was used for every experiment.
- With 65 distinct characters, a character-level model would need 65 output classes. A subword (BPE) tokenizer with up to 500 entries merges frequent character sequences (for example common word pieces), so each window of `N` tokens covers more text.
- Speaker lines end in `:` and a newline; expect to see these as tokens, and possibly in attention patterns later.

**Self-check**

1. Why must the split be made on the raw text *before* the tokenizer is fitted?
2. What would you do if the checksum assertion failed?

**Report prompt:** In Methods, state the source URL, download date, character count, and that the same checksum-verified text was used throughout.


## 1. Data preparation

The raw text is split by position before any tokenizer is fitted or any window is created, so nothing learned from validation text can reach training.


**Coming in Step 1:** split first 80% / final 20% by character position; fit BPE (≤ 500 entries) on training text only and explain unseen symbols; build `N + 1`-token windows (input `z[0:N]`, target `z[1:N+1]`); overlapping training windows; stride-`N` validation windows; no window crosses the split.

**Preview questions:** Why are the targets shifted by one token? What would leak if a window straddled the split?


## 2. Tiny decoder-only Transformer

The model is built from token embeddings, sinusoidal positional encodings, causal multi-head self-attention written with tensor operations, RMSNorm, residual connections, a feed-forward network, and a vocabulary projection.


**Coming in Step 2:** token embeddings plus sinusoidal positions; causal multi-head attention from basic layers; RMSNorm, residuals, feed-forward, output projection; causality checks in `eval()`; cross-entropy on raw logits; baseline plus one controlled change under a fixed step budget.

**Preview questions:** What prevents a token from seeing the future? Why must softmax not be applied before `cross_entropy`?


## 3. Training curves, perplexity, and attention

All evaluation runs in `model.eval()` under `torch.no_grad()`. Validation loss is the total negative log-likelihood divided by the number of scored target tokens, and perplexity is its exponential.


**Coming in Step 3:** training and validation cross-entropy curves; validation loss as total NLL over scored tokens and `PPL = exp(loss)`; attention heatmaps for at least two heads or layers on one fixed validation sequence.

**Preview questions:** Why is averaging batch perplexities wrong? Why does a bright attention cell not prove causal importance?


## 4. Comparison of configurations

The configurations differ in one controlled setting; the split, tokenizer, context length, seed, and training-step budget are fixed.


**Coming in Step 4:** a compact table of configurations, parameter counts, training time, best validation loss, and PPL; discussion of the controlled change, attention patterns, positional encoding and masking, runtime bottlenecks, and limitations.
